In [6]:
import os
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
from groq import Groq
from rag_helper import RAGBase
from openai import OpenAI
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))



# Fetch lesson pages
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]

# Q1 Answer
print(f"Q1: Total documents = {len(documents)}")

documents[0]

Q1: Total documents = 72


{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
print("✓ Index created successfully")

✓ Index created successfully


In [8]:
query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query,
    boost_dict={"content": 3},
    filter_dict={},
    num_results=5
)

print(f"Q2: First result filename = {search_results[0]['filename']}")

Q2: First result filename = 01-agentic-rag/lessons/14-agentic-loop.md


In [9]:
rag = RAGBase(
    index=index,
    llm_client=client,
    model="llama-3.3-70b-versatile"
)


question = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens = rag.rag(question)

print(f"Q3 input tokens: {input_tokens}")

Q3 input tokens: 7221


In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)


In [11]:

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)



rag_chunks = RAGBase(
    index=chunk_index,
    llm_client=client,
    model="llama-3.3-70b-versatile"
)


question = "How does the agentic loop keep calling the model until it stops?"
answer, input_tokens = rag_chunks.rag(question)

print(f"Q3 input tokens: {input_tokens}")

Q3 input tokens: 101


In [13]:
def search(query: str) -> list[dict]:
     """Search the LLM Zoomcamp course content for relevant information.
 
     Args:
         query: The search query string to look up in the course materials.
 
     Returns:
         A list of up to 5 relevant document chunks, each with 'filename' and 'content'.
     """
     results = chunk_index.search(
         query,
         boost_dict={"content": 3},
         filter_dict={},
         num_results=5,
     )
     return [{"filename": r["filename"], "content": r["content"]} for r in results]

In [18]:

 
tools = Tools()
tools.add_tool(search)
 
groq_client = OpenAI(
     api_key=os.getenv("GROQ_API_KEY"),
     base_url="https://api.groq.com/openai/v1",
 )
llm_client = OpenAIChatCompletionsClient(
     model="llama-3.3-70b-versatile",
     client=groq_client,
 )
 
INSTRUCTIONS = (
     "You're a course teaching assistant. Answer the student's question using "
     "the search tool. Make multiple searches with different keywords before answering."
 )
 
runner = OpenAIChatCompletionsRunner(
     tools=tools,
     developer_prompt=INSTRUCTIONS,
     llm_client=llm_client,
 )
 
question = "How does the  agentic loop work, and how is it different from plain RAG?"
result = runner.loop(question)
 
search_call_count = sum(
     1 for msg in result.all_messages
     if isinstance(msg, dict) and msg.get("role") == "tool"
 )
 
print(result.last_message)
print(f"\n>>> Search tool called {search_call_count} time(s).")

The agentic loop is an extension of the Retrieve, Augment, Generate (RAG) framework that enables more flexible and dynamic interaction between the retrieval and generation components. Unlike plain RAG, which uses a fixed set of retrieved documents to generate a response, the agentic loop allows the model to iteratively refine its retrieval and generation based on the current context and user input. This is achieved through a feedback loop where the generated response is used to update the retrieval query, which in turn retrieves new documents that are used to generate a revised response. The agentic loop enables more nuanced and context-dependent generation, as the model can adapt its retrieval and generation strategy based on the evolving conversation.

>>> Search tool called 3 time(s).
